# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimrafarooq2006-coder/ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal 1: staleness (days since last update). Verdict: MIXED.
Pages not updated for 91-180 days decline most often (61.1%, n=9,171)
versus 51.2% for pages updated in the last 90 days (n=20,655). The
181-365 and 365+ groups are too small to trust (n=169 and n=5), so I
cannot say what happens beyond 180 days.

Signal 2: visibility (impressions in 90 days). Verdict: CONFIRMED.
Pages with 1-100 impressions decline much less (38.9%, n=8,006) than
pages with 101+ (54.7% to 62.4%). The rate peaks at 501-5,000
impressions and dips above 5,000, so more views is not always worse.
Staleness is the signal behind FlyRank's refresh flags.

My rule: an aging, visible page deserves review. Aging means 90 or
more days since the last update, and visible means 100 or more
impressions in 90 days. I confirm Lane 2 as my lane.

In [3]:
import os, subprocess
import pandas as pd, numpy as np

path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    if not os.path.isdir("ml-internship-work"):
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/nimrafarooq2006-coder/ml-internship-work"], check=True)
    path = "ml-internship-work/" + path

df = pd.read_csv(path)
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("SIGNAL 1: staleness (days since last update)")
df["stale_bucket"] = pd.cut(df["days_since_last_update"], [-1, 90, 180, 365, 10**6],
                            labels=["0-90", "91-180", "181-365", "365+"])
print(df.groupby("stale_bucket", observed=True)["declining"]
        .agg(n="size", declining_rate="mean").round(3))

print("\nSIGNAL 2: visibility (impressions in 90 days)")
df["volume_bucket"] = pd.cut(df["impressions_90d"], [-1, 0, 100, 500, 5000, 10**9],
                             labels=["0", "1-100", "101-500", "501-5000", "5000+"])
print(df.groupby("volume_bucket", observed=True)["declining"]
        .agg(n="size", declining_rate="mean").round(3))


SIGNAL 1: staleness (days since last update)
                  n  declining_rate
stale_bucket                       
0-90          20655           0.512
91-180         9171           0.611
181-365         169           0.467
365+              5           0.600

SIGNAL 2: visibility (impressions in 90 days)
                   n  declining_rate
volume_bucket                       
1-100           8006           0.389
101-500         5279           0.604
501-5000       10565           0.624
5000+           6150           0.547


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
os.makedirs("work/outputs", exist_ok=True)

aging = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 100).astype(int)
flag = aging * visible

df["score"] = flag * np.log10(df["impressions_90d"].clip(lower=1))
df["reason_code"] = np.where(flag == 1, "aging_visible_page", "no_flag")
df["action"] = np.where(flag == 1, "Review and refresh", "Monitor")

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "impressions_90d", "days_since_last_update",
     "score", "reason_code", "action", "declining"]
].rename(columns={"declining": "declining_label_not_used_in_score"})

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Flagged pages:", int(flag.sum()), "of", len(df))
print("Declining rate, flagged:", round(df.loc[flag == 1, "declining"].mean(), 3))
print("Declining rate, not flagged:", round(df.loc[flag == 0, "declining"].mean(), 3))
queue.head(10)


Flagged pages: 8119 of 30000
Declining rate, flagged: 0.623
Declining rate, not flagged: 0.512


,content_id,client_id,impressions_90d,days_since_last_update,score,reason_code,action,declining_label_not_used_in_score
6653,content_5fe46e04994d,client_4e07408562,517715,104,5.714091,aging_visible_page,Review and refresh,1
29400,content_2dba2b1f9536,client_6208ef0f77,443434,104,5.646829,aging_visible_page,Review and refresh,0
13537,content_2c2606c5d176,client_19581e27de,347399,104,5.540829,aging_visible_page,Review and refresh,1
26531,content_cb112fce36be,client_19581e27de,309910,104,5.491236,aging_visible_page,Review and refresh,1
21565,content_9532f197bbc8,client_4e07408562,309192,104,5.490228,aging_visible_page,Review and refresh,1
3394,content_36ff89c8214e,client_19581e27de,295097,104,5.469965,aging_visible_page,Review and refresh,0
26798,content_b28d1efd668f,client_6208ef0f77,286608,104,5.457288,aging_visible_page,Review and refresh,0
23767,content_813e88069237,client_6208ef0f77,233561,104,5.368400,aging_visible_page,Review and refresh,1
26255,content_c21024970297,client_19581e27de,211366,104,5.325035,aging_visible_page,Review and refresh,0
7445,content_c8e9d6ab9013,client_19581e27de,208678,104,5.319477,aging_visible_page,Review and refresh,1


## 3. Top-20 review

Top 10 review. Action for all ten: Review and refresh. Reason code for
all ten: aging_visible_page.

1. content_5fe46e04994d: 517,715 impressions, 104 days since update.
   Wrong if its traffic is steady and the drop is only seasonal. (Label: declining, so the flag looks right.)
2. content_2dba2b1f9536: 443,434 impressions, 104 days.
   Wrong if it is a stable, high-traffic page that a refresh would only risk. (Label: not declining, so this looks like a false alarm.)
3. content_2c2606c5d176: 347,399 impressions, 104 days.
   Wrong if its impressions come from brand searches that an update cannot change. (Label: declining.)
4. content_cb112fce36be: 309,910 impressions, 104 days.
   Wrong if a similar page on the same site took over its traffic, so it is a merge case, not a refresh case. (Label: declining.)
5. content_9532f197bbc8: 309,192 impressions, 104 days.
   Wrong if the decline is only a normal dip, because 104 days is barely past my 90-day line. (Label: declining.)
6. content_36ff89c8214e: 295,097 impressions, 104 days.
   Wrong if the page is healthy: it is high traffic, but not clearly aging. (Label: not declining, so a false alarm.)
7. content_b28d1efd668f: 286,608 impressions, 104 days.
   Wrong if its traffic is growing and a refresh is unneeded work. (Label: not declining, so a false alarm.)
8. content_813e88069237: 233,561 impressions, 104 days.
   Wrong if the drop comes from changes in the search results page, not from the content. (Label: declining.)
9. content_c21024970297: 211,366 impressions, 104 days.
   Wrong if reviewers' time is better spent elsewhere, because this page is not declining. (Label: not declining, so a false alarm.)
10. content_c8e9d6ab9013: 208,678 impressions, 104 days.
    Wrong if the decline is noise and the real cause is seasonality. (Label: declining.)

Result: 6 of the top 10 are marked declining.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Weak picks: pages 2, 6, 7 and 9 are not marked declining. They are all
very high-traffic pages that are only 104 days old, just past my
90-day cutoff. My score ranks by views only, so the biggest pages
come first even when nothing suggests a problem.

What this baseline cannot do: it never looks at position or CTR, so it
cannot tell a healthy big page from a troubled one. All ten top pages
come from only three clients, so my queue may reflect those clients'
sites more than a general pattern. The "declining" column is shown
only to review the rule. It is never used in the score.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.